# Whitespace Analysis for Specific Lines

This notebook analyzes whitespace around a specific line of text in a compiled PDF resume.

**Features:**
- Compile LaTeX resume to PDF
- Fuzzy match a specific line of text
- Analyze whitespace (margins, line spacing, indentation)
- Visualize the line position and whitespace


In [ ]:
import sys
sys.path.insert(0, '../backend')

from tailor_tom.latex_compiler import compile_latex
import fitz  # PyMuPDF
from io import BytesIO
from difflib import SequenceMatcher
from typing import Dict, List, Optional, Tuple


## Load Resume LaTeX from main.tex


In [ ]:
# Load LaTeX from main.tex
from pathlib import Path

latex_file = Path("../main.tex")
resume_latex = latex_file.read_text(encoding="utf-8")

print(f"✅ Loaded LaTeX from {latex_file}")
print(f"   Length: {len(resume_latex)} characters")
print(f"   Lines: {resume_latex.count(chr(10))} lines")


## Helper Functions for Whitespace Analysis


In [ ]:
def fuzzy_match_text(
    doc: fitz.Document,
    search_text: str,
    threshold: float = 0.6
) -> List[Dict]:
    """Find text blocks that fuzzy match the search text.
    
    Args:
        doc: PyMuPDF document
        search_text: Text to search for (can be partial)
        threshold: Minimum similarity ratio (0.0 to 1.0)
        
    Returns:
        List of matching text blocks with coordinates and metadata
    """
    matches = []
    search_text_lower = search_text.lower().strip()
    
    for page_num in range(len(doc)):
        page = doc[page_num]
        text_dict = page.get_text("dict")
        
        # Extract all text blocks with coordinates
        for block in text_dict.get("blocks", []):
            if "lines" not in block:
                continue
                
            # Combine all lines in this block
            block_text = ""
            block_lines = []
            
            for line in block["lines"]:
                line_text = ""
                line_spans = []
                
                for span in line.get("spans", []):
                    span_text = span.get("text", "").strip()
                    if span_text:
                        line_text += span_text + " "
                        line_spans.append({
                            "text": span_text,
                            "x0": span["bbox"][0],
                            "y0": span["bbox"][1],
                            "x1": span["bbox"][2],
                            "y1": span["bbox"][3],
                            "font": span.get("font", ""),
                            "size": span.get("size", 0),
                        })
                
                if line_text.strip():
                    block_lines.append({
                        "text": line_text.strip(),
                        "spans": line_spans,
                        "bbox": line["bbox"],
                    })
                    block_text += line_text.strip() + "\n"
            
            # Check fuzzy match on the full block or individual lines
            block_text = block_text.strip()
            
            # Try matching the whole block
            if block_text:
                ratio = SequenceMatcher(None, search_text_lower, block_text.lower()).ratio()
                if ratio >= threshold:
                    matches.append({
                        "page": page_num,
                        "text": block_text,
                        "bbox": block["bbox"],
                        "lines": block_lines,
                        "similarity": ratio,
                        "match_type": "block"
                    })
            
            # Also try matching individual lines
            for line in block_lines:
                line_text_lower = line["text"].lower()
                ratio = SequenceMatcher(None, search_text_lower, line_text_lower).ratio()
                if ratio >= threshold:
                    # Check if we already added this as part of a block match
                    if not any(m.get("text", "").lower() == line_text_lower for m in matches):
                        matches.append({
                            "page": page_num,
                            "text": line["text"],
                            "bbox": line["bbox"],
                            "spans": line["spans"],
                            "similarity": ratio,
                            "match_type": "line"
                        })
    
    # Sort by similarity (best matches first)
    matches.sort(key=lambda x: x["similarity"], reverse=True)
    return matches


In [ ]:
def analyze_whitespace_for_line(
    doc: fitz.Document,
    match: Dict,
    margin_threshold: float = 5.0,
    continuation_threshold: float = 15.0  # pixels to consider a line a continuation
) -> Dict:
    """Analyze whitespace around a matched line, handling multi-line bullet points.
    
    Args:
        doc: PyMuPDF document
        match: Match dictionary from fuzzy_match_text
        margin_threshold: Pixels to consider as margin
        continuation_threshold: Max horizontal difference to consider a line a continuation
        
    Returns:
        Dictionary with whitespace metrics
    """
    page_num = match["page"]
    page = doc[page_num]
    
    # Get page dimensions
    page_rect = page.rect
    page_width = page_rect.width
    page_height = page_rect.height
    
    # Get the matched line's bbox
    line_bbox = match["bbox"]
    x0_start, y0_start, x1_start, y1_start = line_bbox
    
    # Find text on the same page
    text_dict = page.get_text("dict")
    
    # Collect all lines with their info
    all_lines = []
    for block in text_dict.get("blocks", []):
        if "lines" not in block:
            continue
        for line in block["lines"]:
            line_bbox_other = line["bbox"]
            line_text = "".join(span.get("text", "") for span in line.get("spans", [])).strip()
            if line_text:
                all_lines.append({
                    "bbox": line_bbox_other,
                    "text": line_text,
                    "x0": line_bbox_other[0],
                    "y0": line_bbox_other[1],
                    "x1": line_bbox_other[2],
                    "y1": line_bbox_other[3],
                })
    
    # Group continuation lines (same bullet point)
    # A continuation line: starts at similar x position (within threshold) and is directly below
    bullet_lines = [{
        "bbox": line_bbox,
        "text": match["text"],
        "x0": x0_start,
        "y0": y0_start,
        "x1": x1_start,
        "y1": y1_start,
    }]
    
    # Find continuation lines below
    # Wrapped lines typically have VERY small vertical gaps (1-3pt, sometimes even less)
    # New bullet points have larger gaps (typically 5pt+)
    max_wrap_gap = 4.0  # Maximum gap for wrapped lines - very tight threshold
    
    current_y_bottom = y1_start
    current_x_start = x0_start
    continuation_distances = []  # Track gaps for debugging
    
    while True:
        continuation_found = None
        min_dist = float('inf')
        
        for line_info in all_lines:
            # Skip lines that are just bullet markers (single character, likely "-")
            line_text_clean = line_info["text"].strip()
            if len(line_text_clean) <= 2 and (line_text_clean == "-" or line_text_clean == "•" or line_text_clean == "*"):
                continue
            
            # Check if this line is below and could be a continuation
            if line_info["y0"] > current_y_bottom:
                dist = line_info["y0"] - current_y_bottom
                x_diff = abs(line_info["x0"] - current_x_start)
                
                # Consider it a continuation ONLY if:
                # 1. It's VERY close below (small vertical gap, < max_wrap_gap, typically 1-3pt for wraps)
                # 2. It starts at similar x position (within continuation_threshold)
                # 3. It's the closest such line
                # 4. Additional check: if gap is > 2pt, verify it's NOT starting a new independent thought
                #    (wrapped lines typically start with lowercase or continue mid-sentence)
                
                is_likely_continuation = True
                
                # Check if this looks like a new bullet point (even for small gaps)
                line_text_for_check = line_info["text"].strip()
                
                # Check if line starts with a capital letter that begins a new independent thought
                if len(line_text_for_check) > 0 and line_text_for_check[0].isupper():
                    # Get the last line of current bullet to check if it ended with punctuation
                    last_bullet_line = bullet_lines[-1]["text"].strip() if bullet_lines else match["text"].strip()
                    
                    # Check if the line starts with an action verb (strong indicator of new bullet)
                    action_starters = ['Identified', 'Led', 'Built', 'Designed', 'Implemented', 'Created', 
                                      'Developed', 'Managed', 'Achieved', 'Increased', 'Improved', 'Reduced',
                                      'Owned', 'Collaborated', 'Spearheaded', 'Directed', 'Engineered', 'Maintained',
                                      'Curated', 'Deployed', 'Optimized', 'Architected', 'Integrated']
                    
                    starts_with_action_verb = any(line_text_for_check.startswith(starter + ' ') for starter in action_starters)
                    
                    # If it starts with an action verb, it's VERY likely a new bullet point
                    # This is a strong signal, regardless of gap size
                    if starts_with_action_verb:
                        is_likely_continuation = False
                    # If gap is > 2pt AND previous line ends cleanly (period or normal word end) AND this starts with capital
                    # it's likely a new bullet point
                    elif dist > 2.0 and last_bullet_line:
                        # If previous line ends with period, it's more likely a new sentence/bullet
                        if last_bullet_line[-1] == '.':
                            is_likely_continuation = False
                        # If previous line doesn't end with punctuation but this starts with capital,
                        # and gap is > 2.5pt, likely new bullet
                        elif dist > 2.5 and not last_bullet_line[-1] in ['.', ',', ';', ':', '-', '—']:
                            is_likely_continuation = False
                
                if dist < max_wrap_gap and x_diff < continuation_threshold and dist < min_dist and is_likely_continuation:
                    min_dist = dist
                    continuation_found = line_info
        
        if continuation_found:
            gap = continuation_found["y0"] - current_y_bottom
            continuation_distances.append(gap)
            bullet_lines.append(continuation_found)
            current_y_bottom = continuation_found["y1"]
            current_x_start = continuation_found["x0"]
        else:
            break
    
    # Calculate combined bbox for all lines in the bullet
    combined_x0 = min(line["x0"] for line in bullet_lines)
    combined_y0 = min(line["y0"] for line in bullet_lines)
    combined_x1 = max(line["x1"] for line in bullet_lines)  # Use the RIGHTMOST x1
    combined_y1 = max(line["y1"] for line in bullet_lines)
    
    # Get the LAST line (for overflow detection - only last line matters)
    last_line = bullet_lines[-1]
    
    # Find the line above (different bullet/item)
    line_above = None
    min_above_dist = float('inf')
    for line_info in all_lines:
        if line_info["y1"] < combined_y0:
            dist = combined_y0 - line_info["y1"]
            # Consider it the line above if it's reasonably close (not too far)
            if dist < min_above_dist and dist < 30:  # Max 30pt gap
                min_above_dist = dist
                line_above = line_info
    
    # Find the line below (different bullet/item)
    # Use same threshold as continuation detection to be consistent
    line_below = None
    min_below_dist = float('inf')
    for line_info in all_lines:
        if line_info["y0"] > combined_y1:
            dist = line_info["y0"] - combined_y1
            # Only consider it if it's NOT a continuation (different x position or larger gap than wraps)
            x_diff = abs(line_info["x0"] - combined_x0)
            if dist < min_below_dist and (x_diff > continuation_threshold or dist > max_wrap_gap):
                min_below_dist = dist
                line_below = line_info
    
    # Calculate whitespace metrics using the FIRST line's left margin
    left_margin = combined_x0 - page_rect.x0
    # Right margin should be calculated from the LAST line (rightmost point)
    right_margin = page_rect.x1 - combined_x1
    total_width = combined_x1 - combined_x0
    text_width_percent = (total_width / page_width) * 100
    
    # Calculate spacing
    spacing_above = min_above_dist if line_above else None
    spacing_below = min_below_dist if line_below else None
    
    # Detect actual page margins by finding leftmost and rightmost text
    # (excluding very short lines that might be bullets or artifacts)
    actual_left_margin = None
    actual_right_margin = None
    for line_info in all_lines:
        line_text_clean = line_info["text"].strip()
        # Skip very short lines (likely bullets/markers)
        if len(line_text_clean) > 3:
            line_x0 = line_info["x0"]
            line_x1 = line_info["x1"]
            if actual_left_margin is None or line_x0 < actual_left_margin:
                actual_left_margin = line_x0
            if actual_right_margin is None or line_x1 > actual_right_margin:
                actual_right_margin = line_x1
    
    # Fallback to page edges if we couldn't detect margins
    if actual_left_margin is None:
        actual_left_margin = page_rect.x0
    if actual_right_margin is None:
        actual_right_margin = page_rect.x1
    
    # Calculate margin sizes (distance from page edge to content)
    detected_left_margin_size = actual_left_margin - page_rect.x0
    detected_right_margin_size = page_rect.x1 - actual_right_margin
    
    # Use detected margins for overflow checking (with threshold)
    overflow_left = combined_x0 < (actual_left_margin - margin_threshold)
    # Only check overflow on the LAST line's right edge
    overflow_right = last_line["x1"] > (actual_right_margin + margin_threshold)
    
    # Combine all text from bullet lines
    full_bullet_text = " ".join(line["text"] for line in bullet_lines)
    
    return {
        "page": page_num,
        "line_text": full_bullet_text,  # Full bullet text including continuations
        "line_count": len(bullet_lines),  # Number of lines in this bullet
        "continuation_gaps": continuation_distances,  # Debug: gaps between continuation lines
        "coordinates": {
            "x0": combined_x0,
            "y0": combined_y0,
            "x1": combined_x1,  # Rightmost point across all lines
            "y1": combined_y1,
            "last_line_x1": last_line["x1"],  # Right edge of last line (for overflow check)
        },
        "margins": {
            "left": left_margin,
            "right": right_margin,
            "left_points": left_margin,
            "right_points": right_margin,
            "left_inches": left_margin / 72.0,
            "right_inches": right_margin / 72.0,
        },
        "line_width": {
            "points": total_width,
            "inches": total_width / 72.0,
            "percent_of_page": text_width_percent,
            "last_line_width": last_line["x1"] - last_line["x0"],
        },
        "spacing": {
            "above_points": spacing_above,
            "below_points": spacing_below,
            "above_inches": spacing_above / 72.0 if spacing_above else None,
            "below_inches": spacing_below / 72.0 if spacing_below else None,
        },
        "overflow": {
            "left": overflow_left,
            "right": overflow_right,
            "has_overflow": overflow_left or overflow_right,
            "detected_left_margin": detected_left_margin_size,
            "detected_right_margin": detected_right_margin_size,
            "actual_left_content": actual_left_margin,
            "actual_right_content": actual_right_margin,
        },
        "page_dimensions": {
            "width_points": page_width,
            "height_points": page_height,
            "width_inches": page_width / 72.0,
            "height_inches": page_height / 72.0,
        },
        "line_above": line_above["text"] if line_above else None,
        "line_below": line_below["text"] if line_below else None,
        "bullet_lines": [line["text"] for line in bullet_lines],  # All lines in the bullet
    }


In [ ]:
def print_whitespace_analysis(analysis: Dict):
    """Pretty print whitespace analysis results."""
    print("=" * 80)
    print("WHITESPACE ANALYSIS")
    print("=" * 80)
    print(f"\nMatched Bullet Text:")
    print(f"  '{analysis['line_text']}'")
    
    if analysis.get('line_count', 1) > 1:
        print(f"\n⚠️  This is a multi-line bullet point ({analysis['line_count']} lines)")
        gaps = analysis.get('continuation_gaps', [])
        if gaps:
            print(f"   Vertical gaps between lines: {[f'{g:.2f}pt' for g in gaps]}")
        print(f"   Lines:")
        for i, line_text in enumerate(analysis.get('bullet_lines', []), 1):
            print(f"     {i}. {line_text}")
    
    print(f"\nPage: {analysis['page'] + 1}")
    print(f"\nCoordinates (points):")
    print(f"  x0: {analysis['coordinates']['x0']:.2f}, y0: {analysis['coordinates']['y0']:.2f}")
    print(f"  x1: {analysis['coordinates']['x1']:.2f}, y1: {analysis['coordinates']['y1']:.2f}")
    
    print(f"\nMargins:")
    print(f"  Left:  {analysis['margins']['left_points']:.2f} pt ({analysis['margins']['left_inches']:.2f} in)")
    print(f"  Right: {analysis['margins']['right_points']:.2f} pt ({analysis['margins']['right_inches']:.2f} in)")
    
    print(f"\nLine Width:")
    print(f"  {analysis['line_width']['points']:.2f} pt ({analysis['line_width']['inches']:.2f} in)")
    print(f"  {analysis['line_width']['percent_of_page']:.1f}% of page width")
    
    print(f"\nSpacing:")
    if analysis['spacing']['above_points'] is not None:
        print(f"  Above: {analysis['spacing']['above_points']:.2f} pt ({analysis['spacing']['above_inches']:.2f} in)")
    else:
        print(f"  Above: No line found above")
    
    if analysis['spacing']['below_points'] is not None:
        print(f"  Below: {analysis['spacing']['below_points']:.2f} pt ({analysis['spacing']['below_inches']:.2f} in)")
    else:
        print(f"  Below: No line found below")
    
    print(f"\nOverflow Detection:")
    overflow_info = analysis['overflow']
    print(f"  Detected page margins:")
    print(f"    Left margin size:  {overflow_info.get('detected_left_margin', 'N/A'):.2f} pt ({overflow_info.get('detected_left_margin', 0) / 72.0:.2f} in)")
    print(f"    Right margin size: {overflow_info.get('detected_right_margin', 'N/A'):.2f} pt ({overflow_info.get('detected_right_margin', 0) / 72.0:.2f} in)")
    print(f"  Content boundaries:")
    print(f"    Leftmost content:  {overflow_info.get('actual_left_content', 'N/A'):.2f} pt")
    print(f"    Rightmost content: {overflow_info.get('actual_right_content', 'N/A'):.2f} pt")
    if analysis.get('line_count', 1) > 1:
        print(f"  Note: For multi-line bullets, overflow is checked on the LAST line only")
        print(f"  Last line right edge: {analysis['coordinates'].get('last_line_x1', 'N/A'):.2f} pt")
    print(f"  Left overflow:  {analysis['overflow']['left']} (line starts at {analysis['coordinates']['x0']:.2f} pt)")
    print(f"  Right overflow: {analysis['overflow']['right']} (last line ends at {analysis['coordinates'].get('last_line_x1', analysis['coordinates']['x1']):.2f} pt)")
    print(f"  Has overflow:   {analysis['overflow']['has_overflow']}")
    
    if analysis['line_above']:
        print(f"\nLine Above:")
        print(f"  '{analysis['line_above']}'")
    
    if analysis['line_below']:
        print(f"\nLine Below:")
        print(f"  '{analysis['line_below']}'")
    
    print("\n" + "=" * 80)


## Compile LaTeX to PDF


In [ ]:
# Compile the LaTeX
print("Compiling LaTeX to PDF...")
compile_result = compile_latex(resume_latex)

if not compile_result.success:
    print(f"❌ Compilation failed: {compile_result.error_message}")
    raise Exception("Compilation failed")

print(f"✅ Compilation successful! ({compile_result.page_count} page(s))")

# Open the PDF
pdf_bytes = compile_result.pdf_bytes
doc = fitz.open(stream=pdf_bytes, filetype="pdf")
print(f"📄 PDF opened: {len(doc)} page(s)")


## Search for a Specific Line

Enter the text you want to analyze (partial text is OK - it will fuzzy match):


In [ ]:
# Enter the text you want to analyze
# Note: PDF text won't have LaTeX commands like \textbf{}, so search for plain text
# This version has ". This is to give it whitespace." added to make it 3 lines with lots of whitespace
# Using a shorter, unique search string to better match the full bullet
search_text = "Led a team of 4 to develop a Large Language Model (LLM) utilizing Python, PyTorch, and HDBScan to conduct AI-driven cluster"

# Fuzzy match (lower threshold = more matches, higher = stricter)
threshold = 0.5

print(f"Searching for: '{search_text}'")
print(f"Similarity threshold: {threshold}\n")

matches = fuzzy_match_text(doc, search_text, threshold=threshold)

if not matches:
    print("❌ No matches found. Try:")
    print("  - Lowering the threshold (e.g., 0.3)")
    print("  - Using a longer search phrase")
    print("  - Using partial text from the line")
else:
    print(f"✅ Found {len(matches)} match(es):\n")
    
    for i, match in enumerate(matches[:5]):  # Show top 5
        print(f"Match {i+1} (similarity: {match['similarity']:.2%}):")
        print(f"  Page {match['page'] + 1}")
        print(f"  Text: '{match['text']}'")
        print()


## Analyze Whitespace for the Best Match

Select which match to analyze (0 = best match, 1 = second best, etc.):


In [ ]:
# Select which match to analyze (0 = best match)
match_index = 0

if not matches:
    raise Exception("No matches found. Run the search cell first.")

if match_index >= len(matches):
    raise Exception(f"Match index {match_index} out of range. Found {len(matches)} matches.")

selected_match = matches[match_index]
print(f"Analyzing match {match_index + 1}:")
print(f"  Similarity: {selected_match['similarity']:.2%}")
print(f"  Text: '{selected_match['text']}'")
print()

# Analyze whitespace
analysis = analyze_whitespace_for_line(doc, selected_match)

# Print results
print_whitespace_analysis(analysis)

# Print what would be sent to the LLM as layout feedback
print("\n" + "=" * 80)
print("LAYOUT FEEDBACK (What would be sent to LLM)")
print("=" * 80)

# Simulate the layout feedback format
overflow_info = analysis.get('overflow', {})
current_pages = 1  # Assuming single page
target_pages = 1

feedback_lines = []
feedback_lines.append("LAYOUT ANALYSIS:")
feedback_lines.append("")
feedback_lines.append(f"Page Status: {current_pages} page(s) - FITS within target of {target_pages} page(s)")
feedback_lines.append("")
feedback_lines.append("Instructions: Actively integrate job keywords while keeping the same length.")
feedback_lines.append("- KEEP all bullet points, skills, and content")
feedback_lines.append("- REPHRASE to add keywords (swap words, don't add length)")
feedback_lines.append("")

# Bullet length info
line_count = analysis.get('line_count', 1)
if line_count > 3:
    feedback_lines.append(f"*** LONG BULLET - MUST FIX ***")
    feedback_lines.append(f"This bullet is too long ({line_count} lines). Condense to 2-3 lines.")
    feedback_lines.append(f"  Bullet: '{analysis['line_text'][:60]}...'")
    feedback_lines.append("")
else:
    feedback_lines.append(f"Bullet Length: {line_count} line(s) - within limit (2-3 lines) - OK")
    feedback_lines.append("")

# Overflow info
has_overflow = overflow_info.get('has_overflow', False)
if has_overflow:
    feedback_lines.append("*** OVERFLOW ISSUES - MUST FIX ***")
    if overflow_info.get('left', False):
        feedback_lines.append(f"  Left overflow: Line starts at {analysis['coordinates']['x0']:.1f}pt (content boundary: {overflow_info.get('actual_left_content', 0):.1f}pt)")
    if overflow_info.get('right', False):
        last_x1 = analysis['coordinates'].get('last_line_x1', analysis['coordinates']['x1'])
        feedback_lines.append(f"  Right overflow: Last line ends at {last_x1:.1f}pt (content boundary: {overflow_info.get('actual_right_content', 0):.1f}pt)")
    feedback_lines.append("  ACTION REQUIRED: Shorten this bullet - remove items or use shorter words")
    feedback_lines.append("")
else:
    feedback_lines.append("Overflow Status: No overflow detected - all text within margins")
    feedback_lines.append("")

# Whitespace info
left_margin = analysis['margins']['left_points']
right_margin = analysis['margins']['right_points']
feedback_lines.append(f"Whitespace:")
feedback_lines.append(f"  Left margin: {left_margin:.1f} pt ({left_margin/72.0:.2f} in)")
feedback_lines.append(f"  Right margin: {right_margin:.1f} pt ({right_margin/72.0:.2f} in)")
feedback_lines.append(f"  Line width: {analysis['line_width']['points']:.1f} pt ({analysis['line_width']['percent_of_page']:.1f}% of page)")

print("\n".join(feedback_lines))
print("=" * 80)


## Visualize the Line Position (Optional)

Create a visual representation showing the line position and margins:


In [ ]:
try:
    from matplotlib import pyplot as plt
    import matplotlib.patches as patches
    
    # Create visualization
    fig, ax = plt.subplots(1, 1, figsize=(12, 8))
    
    page_num = analysis['page']
    page = doc[page_num]
    page_rect = page.rect
    
    # Draw page outline
    page_rect_vis = patches.Rectangle(
        (page_rect.x0, page_rect.y0),
        page_rect.width,
        page_rect.height,
        linewidth=2,
        edgecolor='black',
        facecolor='white'
    )
    ax.add_patch(page_rect_vis)
    
    # Draw detected margins (from analysis)
    overflow_info = analysis.get('overflow', {})
    detected_left_margin = overflow_info.get('detected_left_margin', 36.0)
    detected_right_margin = overflow_info.get('detected_right_margin', 41.46)
    actual_left_content = overflow_info.get('actual_left_content', page_rect.x0 + detected_left_margin)
    actual_right_content = overflow_info.get('actual_right_content', page_rect.x1 - detected_right_margin)
    
    # Draw detected content boundaries
    margin_rect = patches.Rectangle(
        (actual_left_content, page_rect.y0 + detected_left_margin),
        actual_right_content - actual_left_content,
        page_rect.height - 2*detected_left_margin,
        linewidth=1,
        edgecolor='gray',
        linestyle='--',
        facecolor='none',
        label='Detected Content Area'
    )
    ax.add_patch(margin_rect)
    
    # Draw vertical lines for content boundaries
    ax.axvline(x=actual_left_content, color='gray', linestyle=':', linewidth=1, alpha=0.5, label='Left Content Boundary')
    ax.axvline(x=actual_right_content, color='gray', linestyle=':', linewidth=1, alpha=0.5, label='Right Content Boundary')
    
    # Draw the matched line
    coords = analysis['coordinates']
    line_rect = patches.Rectangle(
        (coords['x0'], coords['y0']),
        coords['x1'] - coords['x0'],
        coords['y1'] - coords['y0'],
        linewidth=2,
        edgecolor='red',
        facecolor='yellow',
        alpha=0.3
    )
    ax.add_patch(line_rect)
    
    # Draw left margin
    left_margin_rect = patches.Rectangle(
        (page_rect.x0, coords['y0']),
        analysis['margins']['left_points'],
        coords['y1'] - coords['y0'],
        linewidth=1,
        edgecolor='blue',
        facecolor='lightblue',
        alpha=0.2
    )
    ax.add_patch(left_margin_rect)
    
    # Draw right margin
    right_margin_rect = patches.Rectangle(
        (coords['x1'], coords['y0']),
        analysis['margins']['right_points'],
        coords['y1'] - coords['y0'],
        linewidth=1,
        edgecolor='green',
        facecolor='lightgreen',
        alpha=0.2
    )
    ax.add_patch(right_margin_rect)
    
    # Set axis limits
    ax.set_xlim(page_rect.x0 - 20, page_rect.x1 + 20)
    ax.set_ylim(page_rect.y1 + 20, page_rect.y0 - 20)  # Flip Y axis
    ax.set_aspect('equal')
    ax.set_title(f'Line Position and Whitespace Analysis (Page {page_num + 1})')
    ax.set_xlabel('X position (points)')
    ax.set_ylabel('Y position (points)')
    
    # Add legend
    from matplotlib.lines import Line2D
    legend_elements = [
        Line2D([0], [0], color='black', linewidth=2, label='Page boundary'),
        Line2D([0], [0], color='gray', linewidth=1, linestyle='--', label=f'Detected margins ({detected_left_margin:.1f}pt left, {detected_right_margin:.1f}pt right)'),
        Line2D([0], [0], color='gray', linewidth=1, linestyle=':', alpha=0.5, label='Content boundaries'),
        patches.Patch(facecolor='yellow', alpha=0.3, edgecolor='red', label='Matched bullet'),
        patches.Patch(facecolor='lightblue', alpha=0.2, edgecolor='blue', label='Left margin'),
        patches.Patch(facecolor='lightgreen', alpha=0.2, edgecolor='green', label='Right margin'),
    ]
    ax.legend(handles=legend_elements, loc='upper right')
    
    plt.tight_layout()
    plt.show()
    
    print(f"\n📊 Visualization created!")
    print(f"   Red/yellow box = matched bullet point")
    print(f"   Blue box = left margin ({analysis['margins']['left_points']:.1f} pt)")
    print(f"   Green box = right margin ({analysis['margins']['right_points']:.1f} pt)")
    print(f"   Dashed gray rectangle = detected content area")
    print(f"   Dotted gray lines = content boundaries (left: {actual_left_content:.1f}pt, right: {actual_right_content:.1f}pt)")
    
except ImportError:
    print("⚠️  matplotlib not installed. Install with: pip install matplotlib")
except Exception as e:
    print(f"⚠️  Could not create visualization: {e}")


In [ ]:
import json
from pathlib import Path

# Export to JSON
output_dir = Path("../output")
output_dir.mkdir(exist_ok=True)

output_file = output_dir / "whitespace_analysis.json"
with open(output_file, "w") as f:
    json.dump(analysis, f, indent=2)

print(f"✅ Analysis saved to: {output_file}")
